In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# 1. Simulation Parameters

g = 9.81  # Gravity (m/s^2)
v0 = 800.0  # Initial bullet velocity (m/s)
angle_deg = 35.0  # Firing angle in degrees
dt = 0.05  # Time step size (seconds)

# Convert angle to radians

angle_rad = np.radians(angle_deg)

# 2. Matrix Initialization
# State Vector: [x_position, y_position, x_velocity, y_velocity]

state = np.array([
    [0.0],  # Initial X position
    [1.5],  # Initial Y position (fired from shoulder height)
    [v0 * np.cos(angle_rad)],  # Initial X velocity
    [v0 * np.sin(angle_rad)]  # Initial Y velocity
])

# Transition Matrix (A): Maps current state to next state based on dt

A = np.array([
    [1.0, 0.0, dt, 0.0],
    [0.0, 1.0, 0.0, dt],
    [0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 0.0, 1.0]
])

# Control Vector (B): Applies constant deceleration from gravity to velocities
# Control Matrix (u): Contains the gravity constant

B = np.array([
    [0.0],
    [-0.5 * (dt**2)],
    [0.0],
    [-dt]
])

u = g  # Gravity control scalar

# 3. Calculate Trajectory Data for Plotting

x_points = [state[0, 0]]
y_points = [state[1, 0]]

# Loop until the bullet hits the ground (y <= 0)

while state[1, 0] >= 0:
    # State space equation: X_next = A * X_current + B * u
    state = np.dot(A, state) + B * u

    # Store history for the animation
    x_points.append(state[0, 0])
    y_points.append(state[1, 0])

# 4. Set up the Matplotlib Animation Plot

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(0, max(x_points) * 1.05)
ax.set_ylim(0, max(y_points) * 1.1)
ax.set_xlabel("Distance (meters)")
ax.set_ylabel("Height (meters)")
ax.set_title(f"Bullet Trajectory Simulation Matrix Method ({v0} m/s at {angle_deg}°)")
ax.grid(True)

# Visual elements to update

line, = ax.plot([], [], 'b-', label='Bullet Path')
bullet, = ax.plot([], [], 'ro', markersize=6, label='Bullet')
ax.legend()

# Animation initialization function

def init():
    line.set_data([], [])
    bullet.set_data([], [])
    return line, bullet

# Animation frame update function

def update(frame):
    # Draw path up to the current frame
    line.set_data(x_points[:frame], y_points[:frame])
    # Draw current bullet position
    bullet.set_data([x_points[frame]], [y_points[frame]])
    return line, bullet

# 5. Render Animation in Google Colab
# Create the animation object

ani = FuncAnimation(
    fig,
    update,
    frames=len(x_points),
    init_func=init,
    blit=True,
    interval=20
)

# Prevent static plot from showing underneath the video
plt.close()

# Convert the animation into HTML5 video format for Colab rendering
HTML(ani.to_html5_video())